<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B06%5D%20-%20Workshop%20Clustering/Workshop_Clustering_Spotify_RESUELTO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop · Clustering end-to-end con Spotify - VERSIÓN RESUELTA

## El encargo

Sois el equipo de Data Science de Spotify. Producto quiere lanzar **playlists automáticas por tipo de canción**, sin etiquetar nada a mano. Piden tres entregables:

1. **Una segmentación del catálogo**: grupos de canciones que se parecen entre sí, con método y número de grupos justificados.
2. **Una ficha de segmentos**: qué es cada grupo, un nombre que producto entienda y una acción por grupo.
3. **Un mecanismo de asignación**: mañana entran canciones nuevas al catálogo y hay que colocarlas en su segmento sin reentrenar nada.

No hay target: nadie os dirá cuáles son los grupos "correctos". Esto es **aprendizaje no supervisado** de principio a fin.

> **Nota del profesor:** el código y las respuestas son UNA solución razonable, no LA solución. Los resultados exactos varían con la muestra, la semilla y las decisiones de cada equipo.


## El plan de trabajo de ML no supervisado (guárdatelo)

Este plan sirve para **cualquier proyecto no supervisado**: segmentar clientes, detectar anomalías, agrupar productos... Hoy lo recorremos entero con canciones.

| # | Fase | Pregunta que respondes | Entregable |
|---|------|------------------------|------------|
| 1 | Problema de negocio | ¿Qué decisión va a tomar alguien con esto? | Objetivo + unidad de análisis (aquí: la canción) |
| 2 | EDA | ¿Qué datos tengo y en qué estado están? | Lista de problemas y decisiones sobre los datos |
| 3 | Ingeniería de variables | ¿Qué variables entran al modelo y en qué forma? | Matriz de datos limpia, transformada y **escalada** |
| 4 | Reducción de dimensión | ¿Puedo comprimir sin perder señal? ¿Cómo se ven mis datos? | Nº de componentes + visualización 2D (PCA) |
| 5 | Modelado | ¿Qué algoritmo y cuántos grupos? | Modelos + K justificado (codo, silhouette, dendrograma...) |
| 6 | Comparación | ¿Qué solución es mejor **para este caso**? | Modelo elegido y por qué |
| 7 | Profiling | ¿Qué es cada grupo y qué hacemos con él? | Ficha de segmentos: nombre + acción |
| 8 | Activación | ¿Cómo lo usa el producto mañana? | Asignación de datos nuevos (aquí: KNN) |

Fíjate en el orden: **escalar antes de medir distancias**, **reducir antes de visualizar**, **perfilar antes de accionar**. Cambiar el orden es la fuente de errores más habitual.


## Cómo funciona el workshop

- **2,5 horas, 6 fases.** Cada fase tiene un tiempo orientativo y termina en un **✅ Checkpoint**: unas preguntas de control para decidir si puedes cerrar la fase y avanzar.
- El código lo escribes tú: aquí solo hay estructura, tareas y preguntas. Buscar en la documentación (o en tus notebooks de clase) es parte del ejercicio.
- Si vas justo de tiempo, **cierra la fase y avanza**: es mejor un end-to-end completo que una fase perfecta.


---
# Fase 0 · Setup y datos (10 min)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors


In [ ]:
# Configuración recomendada
RANDOM_STATE = 42

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
# Dataset de Spotify: 114.000 canciones con features de audio.
# Si la descarga remota falla, baja el zip a mano y lee desde local.
DATA_URL = "https://drive.google.com/uc?export=download&id=1a9pA0dFbwVH2i6UZjgwcicvcD_SCmG3I"

df_full = pd.read_csv(DATA_URL, compression="zip")
df_full.head(3)


In [ ]:
# Preparación del terreno (dado, para que todo corra rápido en clase):
# - la misma canción aparece repetida en varios géneros -> una por track_id
# - trabajamos con una muestra de 5.000 canciones
# - reservamos 500 canciones "que llegarán mañana al catálogo": NO las toques hasta la Fase 6
df_full = (df_full
           .drop(columns=["Unnamed: 0"])
           .drop_duplicates(subset="track_id")
           .dropna()
           .reset_index(drop=True))

muestra = df_full.sample(n=5500, random_state=RANDOM_STATE)
df_work = muestra.iloc[:5000].reset_index(drop=True)   # tu dataset de trabajo
df_new  = muestra.iloc[5000:].reset_index(drop=True)   # canciones nuevas para la Fase 6

print(f"Catálogo: {df_full.shape[0]} | df_work: {df_work.shape} | df_new: {df_new.shape}")


### Diccionario de datos

| Columna | Qué es |
|---|---|
| `track_id`, `track_name`, `artists`, `album_name` | Identificadores y texto |
| `popularity` | Popularidad 0-100 (calculada por Spotify) |
| `duration_ms` | Duración en milisegundos |
| `explicit` | Contenido explícito (True/False) |
| `danceability`, `energy`, `valence` | Cómo de bailable, enérgica y "alegre" suena (0-1) |
| `acousticness`, `instrumentalness`, `speechiness`, `liveness` | Prob. de ser acústica, instrumental, hablada, en directo (0-1) |
| `loudness` | Volumen medio en dB (negativo: cuanto más cerca de 0, más fuerte) |
| `tempo` | Velocidad en BPM |
| `key`, `mode`, `time_signature` | Tonalidad (0-11), modo mayor/menor, compás |
| `track_genre` | Género asignado a la pista |


---
# Fase 1 · EDA e ingeniería de variables (25 min)

**Objetivo:** decidir qué variables entran al clustering y dejarlas listas en una matriz escalada.

### Tareas
1. Explora `df_work`: tipos, descriptivo, distribuciones, correlaciones.
2. Decide qué variables entran al clustering y cuáles quedan fuera (y por qué).
3. Transforma lo que lo necesite (recuerda IV I: colas largas, variables circulares...).
4. Escala las variables. Guarda el escalador: lo vas a reutilizar en la Fase 6.

### Preguntas guía
- ¿Qué columnas son identificadores o texto que no deben entrar?
- ¿Debe entrar `popularity`? ¿Describe cómo **suena** la canción o cómo le **fue**?
- ¿Y `track_genre`? ¿Qué pasa si agrupas usando (casi) un target?
- ¿Qué variables tienen escalas muy distintas o colas largas?


In [ ]:
df_work.info()
df_work.describe().T[["mean", "std", "min", "max"]].round(2)


In [ ]:
cols_numericas = ["popularity", "duration_ms", "danceability", "energy", "loudness",
                  "speechiness", "acousticness", "instrumentalness", "liveness",
                  "valence", "tempo"]

df_work[cols_numericas].hist(bins=40, figsize=(14, 8), layout=(3, 4))
plt.suptitle("Distribuciones: busca escalas distintas y colas largas")
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 7))
sns.heatmap(df_work[cols_numericas].corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlaciones entre variables numéricas")
plt.show()


In [ ]:
# Decisiones (cada una es discutible; lo importante es justificarla):
# - Fuera identificadores y texto: track_id, track_name, artists, album_name
# - Fuera popularity: no describe cómo suena la canción sino cómo le fue -> la guardamos para el profiling
# - Fuera track_genre: es (casi) un target -> lo reservamos como validación externa en la Fase 5
# - Fuera time_signature: casi todo es 4/4, apenas aporta varianza
# - Fuera mode y explicit: binarias; podrían entrar, pero centramos el clustering en el sonido continuo
# - duration_ms tiene cola larga -> minutos + log1p
# - key es circular (la tonalidad 11 es vecina de la 0): si entrara, iría como sin/cos (IV I);
#   hoy la dejamos fuera para no complicar el perfil

df_work["duration_min_log"] = np.log1p(df_work["duration_ms"] / 60000)

features = ["danceability", "energy", "loudness", "speechiness", "acousticness",
            "instrumentalness", "liveness", "valence", "tempo", "duration_min_log"]

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(df_work[features]), columns=features)

X_scaled.describe().T[["mean", "std", "min", "max"]].round(2)


### Mini reflexión
Responde brevemente en markdown:

- ¿Qué variables has dejado fuera y por qué?
- ¿Qué le pasaría a la distancia entre dos canciones si no escalas?

#### Respuesta:

- Fuera identificadores y texto (no son features), `popularity` (mide éxito, no sonido: si entra, agrupamos por popularidad y el encargo pide tipos de canción) y `track_genre` (es casi un target: agrupar con él sería hacer trampas; mejor reservarlo para validar al final). `time_signature` apenas varía y `duration_ms` entra transformada con log1p por su cola larga.
- Sin escalar, `tempo` (0-240) y `loudness` (-50 a 4) dominarían la distancia euclídea y `danceability` (0-1) no pintaría nada: dos canciones idénticas salvo el tempo quedarían lejísimos. Escalar pone a todas las variables a hablar con el mismo volumen.


### ✅ Checkpoint de fase
Antes de avanzar:
- ¿Tienes una lista clara de variables que entran (y sabrías defender cada exclusión)?
- ¿Has tratado las colas largas?
- ¿Tu matriz está escalada y sin nulos?
- ¿Has guardado el escalador para reutilizarlo con datos nuevos?


---
# Fase 2 · PCA: mira tus datos (15 min)

Tienes ~10 dimensiones y los ojos solo soportan 2. Usa PCA (IV II) para dos cosas:

### Tareas
1. Ajusta un PCA sobre tus datos escalados y mira la **varianza explicada** por componente y acumulada. ¿Cuántas componentes necesitas para el 90%?
2. Proyecta las canciones en **2D** y píntalas (son 5.000 puntos: tamaño pequeño y transparencia).

### Preguntas guía
- ¿Por qué hay que escalar antes de PCA?
- ¿Qué mezcla la primera componente? (mira los pesos si te atreves)


In [ ]:
pca = PCA(random_state=RANDOM_STATE).fit(X_scaled)
var = pca.explained_variance_ratio_
var_acum = np.cumsum(var)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].bar(range(1, len(var) + 1), var)
ax[0].set(title="Varianza explicada por componente", xlabel="Componente")
ax[1].plot(range(1, len(var) + 1), var_acum, marker="o")
ax[1].axhline(0.9, ls="--", c="gray")
ax[1].set(title="Varianza explicada acumulada", xlabel="Nº de componentes")
plt.show()

n_90 = int(np.argmax(var_acum >= 0.9)) + 1
print(f"Con {n_90} componentes conservamos el 90% de la varianza (partíamos de {X_scaled.shape[1]})")

X_pca2 = pca.transform(X_scaled)[:, :2]

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], s=5, alpha=0.4)
plt.xlabel(f"PC1 ({var[0]:.0%} var.)")
plt.ylabel(f"PC2 ({var[1]:.0%} var.)")
plt.title("El catálogo proyectado en 2D")
plt.show()

# Los pesos (loadings): PC1 enfrenta energy/loudness contra acousticness -> "intensidad del sonido"
pd.DataFrame(pca.components_[:2].T, index=features, columns=["PC1", "PC2"]).round(2)


### Mini reflexión

- ¿Cuánta varianza conservas en 2D? ¿Es este mapa una foto fiel o un resumen con pérdida?
- ¿Se ven grupos separados o una nube continua? ¿Demuestra eso que no hay clusters?

#### Respuesta:

- PC1+PC2 conservan en torno al 44% de la varianza: es un resumen con pérdida. Dos puntos juntos en el mapa pueden estar lejos en las 10 dimensiones reales (y al revés). Sirve para orientarse, no para sentenciar.
- Se ve una nube más bien continua con zonas densas. Eso no demuestra que no haya clusters: en 2D solo vemos parte de la señal. La música es un continuo; los algoritmos van a trocear ese continuo y aún así los trozos pueden ser útiles para producto.


### ✅ Checkpoint de fase
- ¿Sabes cuántas componentes hacen falta para el 90% de la varianza?
- ¿Tienes la proyección 2D pintada? La vas a reutilizar para colorear los clusters.


---
# Fase 3 · Primer modelo: K-Means (25 min)

Entrenar K-Means son dos líneas; la decisión importante es **elegir K**. Vas a combinar:

- **Codo**: dónde la inercia deja de bajar con ganas.
- **Silhouette**: cómo de bien asignado está cada punto (-1 a 1).
- **Negocio**: ¿cuántos segmentos puede accionar producto de verdad?

Ninguno de los tres manda solo.

### Tareas
1. Entrena K-Means para un rango de K (por ejemplo 2 a 8) con `k-means++`, y pinta codo y silhouette.
2. Elige tu K y entrena el modelo final.
3. Pinta los clusters sobre tu proyección PCA 2D.


In [ ]:
inercias, siluetas = [], []
rango_k = range(2, 9)

for k in rango_k:
    km_k = KMeans(n_clusters=k, init="k-means++", n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
    inercias.append(km_k.inertia_)
    siluetas.append(silhouette_score(X_scaled, km_k.labels_))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(list(rango_k), inercias, marker="o")
ax[0].set(title="Método del codo", xlabel="K", ylabel="Inercia")
ax[1].plot(list(rango_k), siluetas, marker="o")
ax[1].set(title="Silhouette medio", xlabel="K")
plt.show()

pd.DataFrame({"K": list(rango_k), "inercia": np.round(inercias), "silhouette": np.round(siluetas, 3)})


In [ ]:
# El silhouette máximo cae en K=2, pero 2 segmentos ("tranquilas" vs "enérgicas") no dan
# para un producto de playlists. Entre K=4 y K=6 la métrica es casi plana (~0.15), así que
# el dato deja de decidir y decide el negocio: K=5 da segmentos de tamaño razonable y
# perfiles claros. Números + interpretabilidad + acción: así se defiende un K.
k_elegido = 5

km = KMeans(n_clusters=k_elegido, init="k-means++", n_init=10, random_state=RANDOM_STATE).fit(X_scaled)
labels_km = km.labels_
sil_km = silhouette_score(X_scaled, labels_km)

print(f"K = {k_elegido} | silhouette = {sil_km:.3f}")
print("Canciones por cluster:", np.bincount(labels_km))

plt.figure(figsize=(8, 6))
plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=labels_km, cmap="tab10", s=5, alpha=0.5)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title(f"K-Means con K={k_elegido} sobre el mapa PCA")
plt.show()


### Mini reflexión

- ¿El codo y el silhouette apuntan al mismo K? Si no, ¿con qué criterio desempatas?
- Si el silhouette máximo sale en un K muy bajo, ¿por qué podría aun así no ser la mejor entrega para producto?

#### Respuesta:

- Aquí no coinciden: el silhouette máximo está en K=2 (≈0.25) y el codo es suave, sin quiebro claro. El desempate no es estadístico sino de negocio: ¿cuántos segmentos distintos y accionables necesita el caso de uso? Entre K=4 y K=6 el silhouette apenas cambia, señal de que cualquier K de ese rango es defendible.
- K=2 separa "calmado" vs "enérgico": correcto pero pobre; producto no puede montar una estrategia de playlists con 2 cubos. Un buen entregable sacrifica algo de métrica interna a cambio de segmentos con significado, y lo documenta: "elegimos K=5 con silhouette 0.15 frente al 0.25 de K=2 porque...".


### ✅ Checkpoint de fase
- ¿Has probado varios K, no solo uno?
- ¿Has elegido tu K y sabrías defenderlo en una frase (métrica + negocio)?
- ¿Tienes el modelo final entrenado y los clusters pintados sobre el PCA?

Las fases siguientes usan estas etiquetas: si algo cojea, arréglalo ahora.

---
## ☕ Descanso (5 min)


---
# Fase 4 · Segundo modelo: el retador (25 min)

K-Means asume grupos compactos y "redondos". Ponlo a prueba con un retador. **Elige al menos uno** (los tres si vas sobrado):

| Opción | Modelo | Para qué brilla | Aviso práctico |
|---|---|---|---|
| A | **Jerárquico (ward)** | El dendrograma te deja ver la estructura y cortar donde tenga sentido | Es O(n²): usa una submuestra de ~2.000 canciones |
| B | **DBSCAN** | Formas irregulares y detección de ruido (canciones "inclasificables") | En 10 dimensiones las distancias se difuminan: prueba también sobre las primeras componentes del PCA |
| C | **K-Medoids** | El centro de cada grupo es una **canción real** que puedes escuchar | `!pip install kmedoids`; funciona con matriz de distancias precalculada, usa una submuestra |

### Tareas
1. Entrena tu retador y optimiza sus parámetros (corte del dendrograma, `eps` y `min_samples`, K...).
2. Compáralo con K-Means: silhouette, número de grupos, % de ruido si aplica... y tu valoración cualitativa.

⚠️ Juego limpio: el silhouette solo es comparable **sobre los mismos datos y el mismo espacio**. Si usas submuestra o PCA, dilo en tu conclusión. En DBSCAN, calcula el silhouette sin los puntos de ruido (etiqueta -1) e indica el % de ruido.


In [ ]:
# OPCIÓN A · Clustering jerárquico
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Submuestra: el jerárquico es O(n²) y con 5.000 canciones ya sufre
idx_sub = np.random.default_rng(RANDOM_STATE).choice(len(X_scaled), size=2000, replace=False)
X_sub = X_scaled.iloc[idx_sub].to_numpy()

Z = linkage(X_sub, method="ward")

plt.figure(figsize=(12, 4))
dendrogram(Z, truncate_mode="lastp", p=30, show_leaf_counts=True)
plt.title("Dendrograma (ward, truncado a 30 ramas)")
plt.ylabel("Distancia de fusión")
plt.show()

labels_jer = fcluster(Z, t=k_elegido, criterion="maxclust")
sil_jer = silhouette_score(X_sub, labels_jer)
print(f"Jerárquico con K={k_elegido} (submuestra): silhouette {sil_jer:.3f}")
print("Tamaños:", np.bincount(labels_jer)[1:])


In [ ]:
# OPCIÓN B · DBSCAN
# En el espacio original (10D escaladas) DBSCAN lo pasa mal: con eps pequeño todo es ruido
# y con eps grande todo es un único cluster (maldición de la dimensionalidad, IV II).
# Lo aplicamos sobre las 3 primeras componentes del PCA (~57% de la varianza):
X_pca3 = pca.transform(X_scaled)[:, :3]

for eps in [0.3, 0.5, 0.7, 0.9]:
    db_e = DBSCAN(eps=eps, min_samples=15).fit(X_pca3)
    n_c = len(set(db_e.labels_.tolist()) - {-1})
    print(f"eps={eps}: {n_c} clusters, {(db_e.labels_ == -1).mean():.0%} de ruido")

# eps=0.5 da un equilibrio razonable: estructura + un ~14% de canciones "inclasificables"
db = DBSCAN(eps=0.5, min_samples=15).fit(X_pca3)
labels_db = db.labels_
mask = labels_db != -1
sil_db = silhouette_score(X_pca3[mask], labels_db[mask])
print(f"\nDBSCAN eps=0.5: {len(set(labels_db.tolist()) - {-1})} clusters, "
      f"{(~mask).mean():.0%} de ruido, silhouette sin ruido {sil_db:.3f} (¡en el espacio PCA 3D!)")


In [ ]:
# OPCIÓN C · K-Medoids: el centro de cada grupo es una canción real
!pip install -q kmedoids

from kmedoids import KMedoids
from scipy.spatial.distance import pdist, squareform

D = squareform(pdist(X_sub))          # misma submuestra que el jerárquico
kmed = KMedoids(n_clusters=k_elegido, metric="precomputed", random_state=RANDOM_STATE).fit(D)
labels_kmed = kmed.labels_.astype(int)
sil_kmed = silhouette_score(X_sub, labels_kmed)
print(f"K-Medoids con K={k_elegido} (submuestra): silhouette {sil_kmed:.3f}")

# Escucha las canciones-medoide y tendrás medio profiling hecho:
medoides = df_work.iloc[idx_sub[kmed.medoid_indices_]][["track_name", "artists", "track_genre"]].copy()
medoides.insert(0, "cluster", range(k_elegido))
medoides


In [ ]:
comparacion = pd.DataFrame({
    "modelo": ["K-Means (5.000 canciones)", "Jerárquico ward (submuestra 2.000)",
               "DBSCAN eps=0.5 (espacio PCA 3D)", "K-Medoids (submuestra 2.000)"],
    "grupos": [k_elegido, k_elegido, len(set(labels_db.tolist()) - {-1}), k_elegido],
    "silhouette": [round(sil_km, 3), round(sil_jer, 3), round(sil_db, 3), round(sil_kmed, 3)],
    "nota": ["referencia", "estructura parecida a K-Means",
             "sil no comparable (otro espacio); aporta el concepto de ruido",
             "similar a K-Means pero con centros interpretables"],
})
comparacion


### Mini reflexión

- ¿Tu retador ha encontrado una estructura distinta a la de K-Means o básicamente la misma?
- ¿En qué caso de negocio preferirías DBSCAN? ¿Y K-Medoids?

#### Respuesta:

- Con estos datos, jerárquico y K-Medoids encuentran una estructura muy parecida a la de K-Means (los grupos "de verdad" del sonido son bastante convexos). DBSCAN cuenta otra historia: no busca K grupos, separa zonas densas y marca ~14% de canciones como ruido.
- DBSCAN gana cuando el valor está en las anomalías (fraude, sensores) o los grupos tienen formas raras; aquí, ese 14% "inclasificable" sería oro para un equipo de contenido nicho. K-Medoids gana cuando necesitas explicar los grupos a negocio con un ejemplo real ("este segmento es, literalmente, esta canción") o cuando hay outliers que arrastran a los centroides.


### ✅ Checkpoint de fase
- ¿Has entrenado y optimizado al menos un retador?
- ¿Tienes una comparación honesta con K-Means (mismos datos y mismo espacio, o avisándolo)?
- ¿Has decidido con qué modelo sigues a la Fase 5 y por qué?


---
# Fase 5 · Profiling: de números a segmentos (20 min)

Aquí el clustering deja de ser un gráfico bonito y empieza a tener valor. Aplica el profiling en 4 pasos de la clase de Análisis Cluster:

1. **Perfil** de cada grupo: media por variable, sobre las variables ORIGINALES (nadie entiende "energy = 0.7 desviaciones").
2. **Índice vs media global**: divide el perfil de cada grupo entre la media del catálogo. Un 2.0 = "el doble que la media".
3. **Tamaño** de cada grupo: ¿es un segmento o una anécdota?
4. **Nombre y acción**: el entregable de verdad.

Guardaste `track_genre` y `popularity` fuera del modelo: úsalas ahora como **validación externa**. ¿Los géneros dominantes de cada cluster cuadran con su perfil?

⚠️ Ojo con `loudness`: es negativa (dB), su índice se lee al revés (más alto = MÁS silenciosa).


In [ ]:
df_work["cluster"] = labels_km

cols_perfil = features + ["popularity"]
perfil = df_work.groupby("cluster")[cols_perfil].mean()
perfil["n_canciones"] = df_work.groupby("cluster").size()
print(perfil.round(2).to_string())

indice = perfil[cols_perfil] / df_work[cols_perfil].mean()

plt.figure(figsize=(11, 4))
sns.heatmap(indice, annot=True, fmt=".2f", cmap="RdBu_r", center=1)
plt.title("Índice vs media global (1.0 = media del catálogo)")
plt.show()


In [ ]:
for c in sorted(df_work["cluster"].unique()):
    sub = df_work[df_work["cluster"] == c]
    top = ", ".join(f"{g} ({n})" for g, n in sub["track_genre"].value_counts().head(3).items())
    print(f"Cluster {c} · {len(sub)} canciones · popularidad media {sub['popularity'].mean():.0f}")
    print(f"   géneros top: {top}")


### Ficha de segmentos (ejemplo con K=5 y semilla 42; tus perfiles pueden variar)

- **Voz en directo** (speechiness x5.6, liveness x1.7; comedy): separarlo del universo "música" y alimentar la sección de podcasts/comedia, para que no contamine las playlists musicales.
- **Muro de sonido** (energía alta, instrumental, valence bajo; metal y techno): playlists de intensidad ("Gym rage", "Deep focus techno") segmentadas por hora del día.
- **Calma instrumental** (silenciosa, acústica, instrumental; sleep, ambient, new-age): playlists funcionales de dormir/estudiar; candidata a autoplay nocturno.
- **Acústicas de sofá** (acústica x2, energía media, cantada; tango, romance): playlist "Domingo tranquilo" y moods regionales.
- **Fiesta alegre** (bailable, valence x1.5, lo más popular; salsa, dance): playlists sociales y campañas de fin de semana.


### ✅ Checkpoint de fase
- ¿Tienes el perfil y el índice de cada cluster (y los tamaños)?
- ¿La validación externa (géneros) cuadra con los perfiles?
- ¿Cada segmento tiene nombre y una acción? Este es el entregable nº 2 del encargo.


---
# Fase 6 · De cluster a producto: KNN (15 min)

El clustering está entregado... pero mañana entran canciones nuevas al catálogo y hay que asignarlas a un segmento **sin reentrenar nada**. Conecta con la clase de KNN (eager vs lazy): KNN memoriza el catálogo etiquetado y, cuando llega una canción nueva, mira sus K vecinos y vota.

Fíjate en el patrón: el clustering (no supervisado) ha **creado las etiquetas** que ahora un clasificador (supervisado) aprende a asignar. Se usa muchísimo en proyectos reales.

### Tareas
1. Prepara `df_new` (las 500 canciones de la Fase 0) con **exactamente la misma receta** de features que `df_work`. Piensa: ¿el escalador se vuelve a ajustar o se reutiliza? ¿Por qué?
2. Entrena un KNN con tus datos escalados y tus etiquetas de cluster, y asigna segmento a las canciones nuevas.
3. Sanity check: compara la asignación del KNN con la del centroide más cercano (`.predict()` del K-Means). ¿% de acuerdo?

### Extra ⭐ (si vas bien de tiempo): tu recomendador
Con `NearestNeighbors` sobre TODO el catálogo (`df_full`, misma receta de features): busca una canción que te guste por nombre y saca sus 10 vecinas más cercanas. Tu playlist automática. ¿La pondrías en tu Spotify?


In [ ]:
# Misma receta, mismo scaler: solo .transform(). Hacer fit() aquí sería una fuga
# silenciosa: las canciones nuevas redefinirían la escala de todo lo anterior.
df_new["duration_min_log"] = np.log1p(df_new["duration_ms"] / 60000)
X_new_scaled = pd.DataFrame(scaler.transform(df_new[features]), columns=features)

knn = KNeighborsClassifier(n_neighbors=15)
knn.fit(X_scaled, labels_km)
pred_nuevas = knn.predict(X_new_scaled)

print("Canciones nuevas por cluster:")
print(pd.Series(pred_nuevas).value_counts().sort_index())

acuerdo = (pred_nuevas == km.predict(X_new_scaled)).mean()
print(f"\nAcuerdo KNN vs centroide más cercano: {acuerdo:.1%}")
# Un acuerdo alto (~94%) dice que ambos mecanismos son consistentes. KNN además
# seguiría funcionando con clusters de forma rara, donde el centroide engaña.


In [ ]:
# Escalamos TODO el catálogo (~90.000 canciones) con el scaler ya ajustado
df_cat = df_full.copy()
df_cat["duration_min_log"] = np.log1p(df_cat["duration_ms"] / 60000)
X_cat = scaler.transform(df_cat[features])

nn = NearestNeighbors(n_neighbors=11).fit(X_cat)

MI_CANCION = "Blinding Lights"   # cámbiala por la tuya

candidatas = df_cat[df_cat["track_name"].str.contains(MI_CANCION, case=False, na=False)]
semilla = candidatas.sort_values("popularity", ascending=False).index[0]   # la versión más popular
print("Canción semilla:", df_cat.loc[semilla, "track_name"], "·", df_cat.loc[semilla, "artists"])

dist, vecinos = nn.kneighbors(X_cat[[semilla]])
df_cat.iloc[vecinos[0][1:]][["track_name", "artists", "track_genre", "popularity"]]


### Mini reflexión

- Tu recomendador solo usa features de audio. ¿Qué le falta frente al recomendador real de Spotify?

#### Respuesta:

Le falta el comportamiento: qué escuchan juntos los usuarios (filtrado colaborativo), saltos, likes, contexto (hora, dispositivo), letras y metadatos editoriales. Dos canciones pueden sonar parecidas y vivir en mundos distintos. El audio es una señal útil, sobre todo para canciones nuevas sin historial (cold start), pero el sistema real combina varias señales.


### ✅ Checkpoint de fase
- ¿Las canciones nuevas pasaron por la MISMA receta de features (sin volver a ajustar el escalador)?
- ¿Cada canción nueva tiene un segmento asignado?
- ¿El acuerdo con la asignación por centroide te parece razonable? Este es el entregable nº 3.


---
# Bonus con IA · Ponles nombre con un LLM (si sobra tiempo)

Los nombres de la Fase 5 los pusiste tú. Deja que un LLM proponga los suyos **a partir de tu tabla de perfiles** y compara: buen ejemplo de clustering + IA generativa (el modelo no ve las canciones, solo tu resumen agregado).

Toma tu API key gratuita de [OpenRouter](https://openrouter.ai/settings/keys). Si no tienes key, imprime el prompt y pégalo en cualquier chat de IA.


In [ ]:
prompt = f"""Eres analista musical en Spotify. Esta tabla resume {k_elegido} clusters de canciones
como índice frente a la media del catálogo (1.0 = media, 2.0 = el doble, 0.5 = la mitad).
Ojo: loudness es negativa, así que un índice alto significa MÁS silenciosa.

{indice.round(2).to_string()}

Para cada cluster propón: nombre de playlist (máximo 4 palabras), descripción de una frase
y el momento del día para escucharla. Responde en una tabla markdown."""

print(prompt)

API_KEY = "TU-API-KEY"   # https://openrouter.ai/settings/keys

if API_KEY == "TU-API-KEY":
    print("\n(Sin API key: copia el prompt en cualquier chat de IA y compara sus nombres con los tuyos.)")
else:
    from openai import OpenAI
    client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)
    response = client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": prompt}],
    )
    print(response.choices[0].message.content)


---
# Cierre del workshop

### Preguntas finales
1. De las 8 fases del plan de trabajo, ¿cuál te ha llevado más tiempo? ¿Cuál llevaría más tiempo en un proyecto real?
2. ¿Qué harías distinto si en vez de canciones fueran clientes de un banco? ¿Qué cambia y qué no cambia del plan?
3. Si el silhouette "prefería" un K distinto al que entregaste, ¿cómo lo defiendes ante un perfil técnico?
4. ¿Qué habría que monitorizar si este sistema (clustering + KNN) se pone en producción de verdad?

**Respuestas orientativas:**

1. En clase, el modelado. En un proyecto real: el EDA + la ingeniería de variables (y definir el problema, que hoy venía regalado en el encargo).
2. Cambian los datos (RFM, transacciones, canales) y la acción (campañas en vez de playlists); no cambia el plan: escalar, elegir K con métrica + negocio, perfilar, activar.
3. "El silhouette mide geometría, no valor: con K=2 la partición es más limpia pero inaccionable. Entre K=4 y K=6 la métrica es plana, así que la decisión pasa a criterios de producto, y queda documentada."
4. Drift de las features (¿cambia la distribución del catálogo?), tamaño y estabilidad de los segmentos en el tiempo, % de canciones lejos de todo segmento (candidatas a ruido o a un segmento nuevo) y consistencia KNN vs centroide como alarma barata.

### Takeaways
- El plan de trabajo no supervisado es siempre el mismo: datos -> features escaladas -> (PCA) -> modelo + K justificado -> profiling -> activación.
- Las métricas (codo, silhouette) orientan pero no deciden: el K final se defiende con números + interpretabilidad + acción.
- Cada algoritmo asume una forma de grupo: K-Means (compactos), jerárquico (árbol interpretable), DBSCAN (densidad + ruido), K-Medoids (centros reales).
- El profiling es el entregable: un cluster sin nombre y sin acción no genera valor.
- El ciclo se cierra en producción: el clustering crea etiquetas y un clasificador (KNN) las asigna a datos nuevos, con la misma receta de features y sin fugas.
